<a href="https://colab.research.google.com/github/luzmereles/detector-sipap/blob/main/models/modelo2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q xgboost joblib

In [ ]:
from google.colab import files

archivos = files.upload()

Saving train.csv to train.csv
Saving validation.csv to validation.csv


In [ ]:
"""
Entrenamiento y comparación de XGBoost para detección de fraude SPI.

Dataset:
    train.csv       -> entrenamiento
    validation.csv  -> evaluación durante desarrollo

IMPORTANTE:
    test.csv NO se utiliza en esta etapa.

    El test se reservará para la evaluación final,
    una vez seleccionado el modelo y el umbral.

Experimentos:
    Se comparan distintos valores de scale_pos_weight:

        100
        250
        500
        750
        1000
        1250

Features:
    - monto_pyg
    - zscore_monto_segmento
    - monto_fuera_rango_segmento
    - hora_sin
    - hora_cos
    - hora_fuera_rango_segmento
    - geo_fuera_zona_segmento
    - cambio_de_dispositivo

Target:
    es_fraude
"""


# ============================================================
# IMPORTACIONES
# ============================================================

import joblib
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


# ============================================================
# CONFIGURACIÓN
# ============================================================

RUTA_TRAIN = "train.csv"

RUTA_VALIDATION = "validation.csv"

RUTA_MODELO = "xgboost_mejor_modelo.pkl"


FEATURES = [
    "zscore_monto_segmento",
    "monto_fuera_rango_segmento",
    "hora_sin",
    "hora_cos",
    "hora_fuera_rango_segmento",
    "geo_fuera_zona_segmento",
    "cambio_de_dispositivo",
]


TARGET = "es_fraude"


# Valores que vamos a comparar
PESOS = [
    100,
    250,
    500,
    750,
    1000,
    1250,
]


# Umbral utilizado para esta primera comparación.
# Más adelante podemos probar varios umbrales.
UMBRAL = 0.60


# ============================================================
# FUNCIÓN: MOSTRAR DISTRIBUCIÓN
# ============================================================

def mostrar_distribucion(nombre, df):

    total = len(df)

    fraudes = int(
        df[TARGET].sum()
    )

    legitimas = (
        total - fraudes
    )

    tasa = (
        fraudes / total * 100
        if total > 0
        else 0
    )

    print(
        f"{nombre:<12} | "
        f"Total: {total:>9,} | "
        f"Legítimas: {legitimas:>9,} | "
        f"Fraudes: {fraudes:>4,} | "
        f"Tasa: {tasa:.4f}%"
    )


# ============================================================
# FUNCIÓN: EVALUAR MODELO
# ============================================================

def evaluar_modelo(
    y_real,
    probabilidades,
    umbral
):

    # Convertir probabilidades en predicciones
    predicciones = (
        probabilidades >= umbral
    ).astype(int)


    # --------------------------------------------------------
    # MÉTRICAS
    # --------------------------------------------------------

    precision = precision_score(
        y_real,
        predicciones,
        zero_division=0,
    )


    recall = recall_score(
        y_real,
        predicciones,
        zero_division=0,
    )


    f1 = f1_score(
        y_real,
        predicciones,
        zero_division=0,
    )


    pr_auc = average_precision_score(
        y_real,
        probabilidades,
    )


    roc_auc = roc_auc_score(
        y_real,
        probabilidades,
    )


    # --------------------------------------------------------
    # MATRIZ DE CONFUSIÓN
    # --------------------------------------------------------

    matriz = confusion_matrix(
        y_real,
        predicciones,
    )


    tn, fp, fn, tp = matriz.ravel()


    return {
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }


# ============================================================
# CARGAR DATASETS
# ============================================================

print("=" * 75)
print("ENTRENAMIENTO INICIAL - XGBOOST")
print("=" * 75)


train = pd.read_csv(
    RUTA_TRAIN
)


validation = pd.read_csv(
    RUTA_VALIDATION
)


print("\nDatasets cargados correctamente.")


mostrar_distribucion(
    "Train",
    train
)


mostrar_distribucion(
    "Validation",
    validation
)


# ============================================================
# VALIDAR COLUMNAS
# ============================================================

columnas_requeridas = (
    FEATURES + [TARGET]
)


faltantes_train = [
    columna
    for columna in columnas_requeridas
    if columna not in train.columns
]


faltantes_validation = [
    columna
    for columna in columnas_requeridas
    if columna not in validation.columns
]


if faltantes_train:

    raise ValueError(
        "Columnas faltantes en train.csv: "
        f"{faltantes_train}"
    )


if faltantes_validation:

    raise ValueError(
        "Columnas faltantes en validation.csv: "
        f"{faltantes_validation}"
    )


print("\nValidación de columnas: OK")


# ============================================================
# X / Y
# ============================================================

X_train = train[FEATURES]

y_train = train[TARGET]


X_validation = validation[FEATURES]

y_validation = validation[TARGET]


# ============================================================
# INFORMACIÓN DEL DESBALANCE
# ============================================================

fraudes_train = int(
    y_train.sum()
)


legitimas_train = (
    len(y_train)
    - fraudes_train
)


ratio = (
    legitimas_train / fraudes_train
    if fraudes_train > 0
    else 0
)


print("\n" + "=" * 75)
print("DESBALANCE DE CLASES")
print("=" * 75)


print(
    f"Legítimas train:          "
    f"{legitimas_train:,}"
)


print(
    f"Fraudes train:            "
    f"{fraudes_train:,}"
)


print(
    f"Ratio legítimas/fraude:   "
    f"{ratio:.2f}:1"
)


print(
    "\nPesos a comparar:"
)


print(
    ", ".join(
        str(peso)
        for peso in PESOS
    )
)


# ============================================================
# ENTRENAMIENTO DE LOS MODELOS
# ============================================================

resultados = []

modelos = {}


print("\n" + "=" * 75)
print("INICIO DE EXPERIMENTOS")
print("=" * 75)


for peso in PESOS:

    print("\n")
    print("-" * 75)

    print(
        f"Entrenando "
        f"scale_pos_weight = {peso}"
    )

    print("-" * 75)


    # --------------------------------------------------------
    # CREAR MODELO
    # --------------------------------------------------------

    modelo = XGBClassifier(

        n_estimators=500,

        max_depth=5,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        objective="binary:logistic",

        eval_metric="aucpr",

        scale_pos_weight=peso,

        random_state=42,

        n_jobs=-1,
    )


    # --------------------------------------------------------
    # ENTRENAR
    # --------------------------------------------------------

    modelo.fit(

        X_train,

        y_train,

        eval_set=[
            (X_validation, y_validation),
        ],

        verbose=False,
    )


    # --------------------------------------------------------
    # PREDICCIONES
    # --------------------------------------------------------

    probabilidades = (

        modelo.predict_proba(
            X_validation
        )[:, 1]

    )


    # --------------------------------------------------------
    # EVALUACIÓN
    # --------------------------------------------------------

    metricas = evaluar_modelo(

        y_real=y_validation,

        probabilidades=probabilidades,

        umbral=UMBRAL,
    )


    # Agregar peso utilizado
    metricas[
        "scale_pos_weight"
    ] = peso


    # Guardar resultados
    resultados.append(
        metricas
    )


    # Guardar modelo
    modelos[peso] = modelo


    # --------------------------------------------------------
    # MOSTRAR RESULTADO
    # --------------------------------------------------------

    print(
        f"PR-AUC:    "
        f"{metricas['PR-AUC']:.4f}"
    )

    print(
        f"ROC-AUC:   "
        f"{metricas['ROC-AUC']:.4f}"
    )

    print(
        f"Precision: "
        f"{metricas['Precision']:.4f}"
    )

    print(
        f"Recall:    "
        f"{metricas['Recall']:.4f}"
    )

    print(
        f"F1:        "
        f"{metricas['F1']:.4f}"
    )

    print(
        f"FP:        "
        f"{metricas['FP']:,}"
    )

    print(
        f"FN:        "
        f"{metricas['FN']:,}"
    )

    print(
        f"TP:        "
        f"{metricas['TP']:,}"
    )


# ============================================================
# TABLA COMPARATIVA
# ============================================================

resultados_df = pd.DataFrame(
    resultados
)


columnas_orden = [
    "scale_pos_weight",
    "PR-AUC",
    "ROC-AUC",
    "Precision",
    "Recall",
    "F1",
    "TN",
    "FP",
    "FN",
    "TP",
]


resultados_df = resultados_df[
    columnas_orden
]


# Ordenar por PR-AUC
resultados_df = resultados_df.sort_values(
    "PR-AUC",
    ascending=False
).reset_index(drop=True)


print("\n" + "=" * 75)
print("COMPARACIÓN DE MODELOS")
print("=" * 75)


display(
    resultados_df
)


# ============================================================
# MEJOR MODELO
# ============================================================

mejor_peso = int(
    resultados_df.iloc[0][
        "scale_pos_weight"
    ]
)


mejor_modelo = modelos[
    mejor_peso
]


print("\n" + "=" * 75)
print("MEJOR MODELO SEGÚN PR-AUC")
print("=" * 75)


print(
    f"scale_pos_weight: "
    f"{mejor_peso}"
)


print(
    f"PR-AUC: "
    f"{resultados_df.iloc[0]['PR-AUC']:.4f}"
)


print(
    f"Precision: "
    f"{resultados_df.iloc[0]['Precision']:.4f}"
)


print(
    f"Recall: "
    f"{resultados_df.iloc[0]['Recall']:.4f}"
)


print(
    f"F1: "
    f"{resultados_df.iloc[0]['F1']:.4f}"
)


print(
    f"FP: "
    f"{int(resultados_df.iloc[0]['FP']):,}"
)


print(
    f"FN: "
    f"{int(resultados_df.iloc[0]['FN']):,}"
)


print(
    f"TP: "
    f"{int(resultados_df.iloc[0]['TP']):,}"
)


# ============================================================
# MATRIZ DE CONFUSIÓN DEL MEJOR MODELO
# ============================================================

probabilidades_mejor_modelo = (

    mejor_modelo.predict_proba(
        X_validation
    )[:, 1]

)


predicciones_mejor_modelo = (

    probabilidades_mejor_modelo
    >= UMBRAL

).astype(int)


matriz_mejor = confusion_matrix(
    y_validation,
    predicciones_mejor_modelo,
)


print("\n" + "=" * 75)
print("MATRIZ DE CONFUSIÓN - MEJOR MODELO")
print("=" * 75)


print(
    matriz_mejor
)


tn, fp, fn, tp = (
    matriz_mejor.ravel()
)


print("\nDetalle:")

print(
    f"TN (legítimas correctas):   "
    f"{tn:,}"
)

print(
    f"FP (legítimas marcadas):    "
    f"{fp:,}"
)

print(
    f"FN (fraudes no detectados): "
    f"{fn:,}"
)

print(
    f"TP (fraudes detectados):    "
    f"{tp:,}"
)


# ============================================================
# REPORTE DE CLASIFICACIÓN
# ============================================================

print("\n" + "=" * 75)
print("REPORTE DE CLASIFICACIÓN - MEJOR MODELO")
print("=" * 75)


print(
    classification_report(
        y_validation,
        predicciones_mejor_modelo,
        target_names=[
            "Legítima",
            "Fraude",
        ],
        zero_division=0,
    )
)


# ============================================================
# IMPORTANCIA DE FEATURES
# ============================================================

print("\n" + "=" * 75)
print("IMPORTANCIA DE FEATURES - MEJOR MODELO")
print("=" * 75)


importancias = (

    pd.Series(
        mejor_modelo.feature_importances_,
        index=FEATURES,
    )

    .sort_values(
        ascending=False
    )

)


for feature, importancia in (
    importancias.items()
):

    print(
        f"{feature:<35} "
        f"{importancia:.6f}"
    )


# ============================================================
# GUARDAR MEJOR MODELO
# ============================================================

joblib.dump(
    mejor_modelo,
    RUTA_MODELO,
)


print("\n" + "=" * 75)
print("MODELO GUARDADO")
print("=" * 75)


print(
    f"Archivo: {RUTA_MODELO}"
)


print(
    f"Peso seleccionado: "
    f"{mejor_peso}"
)


print(
    f"Umbral utilizado: "
    f"{UMBRAL}"
)


# ============================================================
# GUARDAR RESULTADOS DE LOS EXPERIMENTOS
# ============================================================

resultados_df.to_csv(
    "comparacion_scale_pos_weight.csv",
    index=False,
)


print(
    "\nTambién se guardó:"
)

print(
    "comparacion_scale_pos_weight.csv"
)

ENTRENAMIENTO INICIAL - XGBOOST

Datasets cargados correctamente.
Train        | Total:   333,462 | Legítimas:   333,196 | Fraudes:  266 | Tasa: 0.0798%
Validation   | Total:    83,678 | Legítimas:    83,604 | Fraudes:   74 | Tasa: 0.0884%

Validación de columnas: OK

DESBALANCE DE CLASES
Legítimas train:          333,196
Fraudes train:            266
Ratio legítimas/fraude:   1252.62:1

Pesos a comparar:
100, 250, 500, 750, 1000, 1250

INICIO DE EXPERIMENTOS


---------------------------------------------------------------------------
Entrenando scale_pos_weight = 100
---------------------------------------------------------------------------
PR-AUC:    0.7913
ROC-AUC:   0.9993
Precision: 0.5508
Recall:    0.8784
F1:        0.6771
FP:        53
FN:        9
TP:        65


---------------------------------------------------------------------------
Entrenando scale_pos_weight = 250
---------------------------------------------------------------------------
PR-AUC:    0.7901
ROC-AUC:   

,scale_pos_weight,PR-AUC,ROC-AUC,Precision,Recall,F1,TN,FP,FN,TP
0,100,0.791296,0.999323,0.550847,0.878378,0.677083,83551,53,9,65
1,250,0.790118,0.999303,0.537190,0.878378,0.666667,83548,56,9,65
2,500,0.785034,0.999090,0.515625,0.891892,0.653465,83542,62,8,66
3,1000,0.783749,0.999192,0.478261,0.891892,0.622642,83532,72,8,66
4,750,0.782925,0.999057,0.492537,0.891892,0.634615,83536,68,8,66
5,1250,0.780481,0.998968,0.485075,0.878378,0.625000,83535,69,9,65



MEJOR MODELO SEGÚN PR-AUC
scale_pos_weight: 100
PR-AUC: 0.7913
Precision: 0.5508
Recall: 0.8784
F1: 0.6771
FP: 53
FN: 9
TP: 65

MATRIZ DE CONFUSIÓN - MEJOR MODELO
[[83551    53]
 [    9    65]]

Detalle:
TN (legítimas correctas):   83,551
FP (legítimas marcadas):    53
FN (fraudes no detectados): 9
TP (fraudes detectados):    65

REPORTE DE CLASIFICACIÓN - MEJOR MODELO
              precision    recall  f1-score   support

    Legítima       1.00      1.00      1.00     83604
      Fraude       0.55      0.88      0.68        74

    accuracy                           1.00     83678
   macro avg       0.78      0.94      0.84     83678
weighted avg       1.00      1.00      1.00     83678


IMPORTANCIA DE FEATURES - MEJOR MODELO
cambio_de_dispositivo               0.643274
hora_fuera_rango_segmento           0.194829
geo_fuera_zona_segmento             0.081596
zscore_monto_segmento               0.045845
monto_fuera_rango_segmento          0.014751
hora_cos                           

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE SEÑALES POR CLASE
# ============================================================

print("\n" + "=" * 75)
print("DISTRIBUCIÓN DE SEÑALES: LEGÍTIMAS VS FRAUDES")
print("=" * 75)

distribucion_senales = (
    train.groupby("es_fraude")[
        [
            "cambio_de_dispositivo",
            "hora_fuera_rango_segmento",
            "geo_fuera_zona_segmento",
            "monto_fuera_rango_segmento",
        ]
    ]
    .mean()
    .rename(index={
        0: "Legítimas",
        1: "Fraudes",
    })
)

print(
    (distribucion_senales * 100).round(2)
)


DISTRIBUCIÓN DE SEÑALES: LEGÍTIMAS VS FRAUDES
           cambio_de_dispositivo  hora_fuera_rango_segmento  \
es_fraude                                                     
Legítimas                   1.96                       2.01   
Fraudes                    85.34                      54.51   

           geo_fuera_zona_segmento  monto_fuera_rango_segmento  
es_fraude                                                       
Legítimas                     1.97                        0.65  
Fraudes                      67.29                       48.50  


In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# ANALISIS DE COMBINACIONES QUE APRENDE XGBOOST
# ============================================================

FEATURES = [
    "zscore_monto_segmento",
    "monto_fuera_rango_segmento",
    "hora_sin",
    "hora_cos",
    "hora_fuera_rango_segmento",
    "geo_fuera_zona_segmento",
    "cambio_de_dispositivo",
]

# ------------------------------------------------------------
# 1. Cargar validación
# ------------------------------------------------------------

validation = pd.read_csv("validation.csv")

X_validation = validation[FEATURES]
y_validation = validation["es_fraude"]

# ------------------------------------------------------------
# 2. Obtener probabilidades del modelo
# ------------------------------------------------------------

probabilidades = mejor_modelo.predict_proba(X_validation)[:, 1]

validation_analisis = validation.copy()
validation_analisis["probabilidad_fraude"] = probabilidades

# ------------------------------------------------------------
# 3. Crear combinaciones de las señales principales
# ------------------------------------------------------------

SIGNALS = [
    "cambio_de_dispositivo",
    "hora_fuera_rango_segmento",
    "geo_fuera_zona_segmento",
    "monto_fuera_rango_segmento",
]

# Nos aseguramos de que sean enteros 0/1
for col in SIGNALS:
    validation_analisis[col] = validation_analisis[col].astype(int)

# ------------------------------------------------------------
# 4. Agrupar por combinación
# ------------------------------------------------------------

resultado = (
    validation_analisis
    .groupby(SIGNALS)
    .agg(
        casos=("es_fraude", "size"),
        fraudes_reales=("es_fraude", "sum"),
        tasa_fraude_real=("es_fraude", "mean"),
        probabilidad_promedio=("probabilidad_fraude", "mean"),
        probabilidad_maxima=("probabilidad_fraude", "max"),
    )
    .reset_index()
)

# Porcentaje de casos que son fraude
resultado["tasa_fraude_real"] *= 100

# Ordenar por cantidad de señales
resultado["cantidad_señales"] = resultado[SIGNALS].sum(axis=1)

resultado = resultado.sort_values(
    ["cantidad_señales", "probabilidad_promedio"],
    ascending=[True, False]
)

# ------------------------------------------------------------
# 5. Mostrar
# ------------------------------------------------------------

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\n" + "=" * 100)
print("ANÁLISIS DE COMBINACIONES")
print("=" * 100)

print(resultado.to_string(index=False))


ANÁLISIS DE COMBINACIONES
 cambio_de_dispositivo  hora_fuera_rango_segmento  geo_fuera_zona_segmento  monto_fuera_rango_segmento  casos  fraudes_reales  tasa_fraude_real  probabilidad_promedio  probabilidad_maxima  cantidad_señales
                     0                          0                        0                           0  78820               0          0.000000               0.000001             0.000737                 0
                     0                          1                        0                           0   1156               1          0.086505               0.001159             0.995810                 1
                     1                          0                        0                           0   1201               0          0.000000               0.000485             0.231501                 1
                     0                          0                        0                           1    544               0          0.000000      

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# COMPROBAR GENERALIZACIÓN DE COMBINACIONES
# ============================================================

SIGNALS = [
    "cambio_de_dispositivo",
    "hora_fuera_rango_segmento",
    "geo_fuera_zona_segmento",
    "monto_fuera_rango_segmento",
]

# ------------------------------------------------------------
# 1. Copias para no modificar los datasets originales
# ------------------------------------------------------------

train_test = train.copy()
validation_test = validation.copy()

# Asegurar 0/1
for col in SIGNALS:
    train_test[col] = train_test[col].astype(int)
    validation_test[col] = validation_test[col].astype(int)


# ------------------------------------------------------------
# 2. Crear una firma para cada combinación
# ------------------------------------------------------------

def crear_combinacion(df):
    return df[SIGNALS].astype(str).agg("-".join, axis=1)


train_test["combinacion"] = crear_combinacion(train_test)
validation_test["combinacion"] = crear_combinacion(validation_test)


# ------------------------------------------------------------
# 3. Contar combinaciones en TRAIN
# ------------------------------------------------------------

conteo_train = (
    train_test
    .groupby("combinacion")
    .size()
    .rename("casos_train")
)


# ------------------------------------------------------------
# 4. Obtener probabilidades en VALIDATION
# ------------------------------------------------------------

X_validation = validation[FEATURES]

probabilidades = mejor_modelo.predict_proba(X_validation)[:, 1]

validation_test["probabilidad_fraude"] = probabilidades


# ------------------------------------------------------------
# 5. Analizar combinaciones de VALIDATION
# ------------------------------------------------------------

resultado = (
    validation_test
    .groupby("combinacion")
    .agg(
        casos_validation=("es_fraude", "size"),
        fraudes_validation=("es_fraude", "sum"),
        probabilidad_promedio=("probabilidad_fraude", "mean"),
        probabilidad_maxima=("probabilidad_fraude", "max"),
    )
    .reset_index()
)


# ------------------------------------------------------------
# 6. Agregar cantidad que existía en TRAIN
# ------------------------------------------------------------

resultado = resultado.merge(
    conteo_train,
    on="combinacion",
    how="left"
)

resultado["casos_train"] = (
    resultado["casos_train"]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 7. Calcular tasa de fraude
# ------------------------------------------------------------

resultado["tasa_fraude_validation"] = (
    resultado["fraudes_validation"]
    / resultado["casos_validation"]
    * 100
)


# ------------------------------------------------------------
# 8. Marcar combinaciones nuevas
# ------------------------------------------------------------

resultado["combinacion_nueva"] = (
    resultado["casos_train"] == 0
)


# ------------------------------------------------------------
# 9. Ordenar
# ------------------------------------------------------------

resultado = resultado.sort_values(
    ["combinacion_nueva", "casos_train"],
    ascending=[False, True]
)


# ------------------------------------------------------------
# 10. Mostrar
# ------------------------------------------------------------

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\n" + "=" * 120)
print("GENERALIZACIÓN DE COMBINACIONES")
print("=" * 120)

print(resultado.to_string(index=False))


GENERALIZACIÓN DE COMBINACIONES
combinacion  casos_validation  fraudes_validation  probabilidad_promedio  probabilidad_maxima  casos_train  tasa_fraude_validation  combinacion_nueva
    0-1-1-1                 4                   4               0.992998             0.999173            4              100.000000              False
    0-1-0-1                 2                   2               0.996758             0.998650            8              100.000000              False
    1-0-0-1                 1                   1               0.999449             0.999449           17              100.000000              False
    1-1-0-1                 6                   6               0.999847             0.999920           26              100.000000              False
    1-0-1-1                 9                   9               0.999837             0.999931           36              100.000000              False
    1-1-1-1                44                  12               0.6